<a href="https://colab.research.google.com/github/nkayn0410-dev/NCKH-LLM-/blob/main/Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [7]:
!git clone --depth=200 https://github.com/curl/curl.git /content/curl
print("✅ Cloned curl")

fatal: destination path '/content/curl' already exists and is not an empty directory.
✅ Cloned curl


In [8]:
import subprocess, re

def get_fix_commits(repo_path):
    result = subprocess.run(
        ['git', '-C', repo_path, 'log', '--all',
         '--grep=fix', '--grep=bug', '--grep=repair',
         '--grep=patch', '--format=%H|%s|%ci', '--no-merges'],
        capture_output=True, text=True
    )
    commits = []
    for line in result.stdout.strip().split('\n'):
        if '|' in line:
            h, msg, date = line.split('|', 2)
            commits.append({'hash': h, 'msg': msg.lower(), 'date': date})
    return commits

fix_commits = get_fix_commits('/content/curl')
print(f"🔍 Found {len(fix_commits)} fix-related commits")

🔍 Found 56 fix-related commits


In [9]:
import subprocess, json

def get_files_changed(repo_path, commit_hash):
    """Trả về danh sách file đã thay đổi trong commit"""
    result = subprocess.run(
        ['git', '-C', repo_path, 'diff-tree', '--no-commit-id', '-r',
         '--name-only', f'{commit_hash}^1', commit_hash],
        capture_output=True, text=True
    )
    return [f for f in result.stdout.split('\n') if f.strip()]

def get_file_content(repo_path, commit_hash, file_path, is_before=True):
    """Lấy nội dung file: before=True → parent, False → current"""
    ref = f'{commit_hash}^1' if is_before else commit_hash
    result = subprocess.run(
        ['git', '-C', repo_path, 'show', f'{ref}:{file_path}'],
        capture_output=True, text=True
    )
    return result.stdout if result.returncode == 0 else None

samples = []
ext = {'.c', '.h'}  # file C

for c in fix_commits[:200]:  # giới hạn 200 commits mỗi repo
    files = get_files_changed('/content/curl', c['hash'])
    for f in files:
        if not any(f.endswith(e) for e in ext):
            continue
        buggy_code = get_file_content('/content/curl', c['hash'], f, is_before=True)
        fixed_code = get_file_content('/content/curl', c['hash'], f, is_before=False)
        if buggy_code and fixed_code and len(buggy_code) > 50:
            samples.append({
                'project': 'curl',
                'file': f,
                'commit': c['hash'],
                'buggy': buggy_code,
                'fixed': fixed_code,
                'label': 1  # 1 = có lỗi (code TRƯỚC khi fix)
            })

print(f"✅ Extracted {len(samples)} code pairs")

✅ Extracted 210 code pairs


In [11]:
import json, os
for idx, sample in enumerate(samples, start=1):
    sample['id'] = f"{idx:03d}"  # Sẽ sinh ra id
os.makedirs('/content/drive/MyDrive/NCKH_dataset', exist_ok=True)
with open('/content/drive/MyDrive/NCKH_dataset/curl_raw.json', 'w') as f:
     json.dump(samples, f, ensure_ascii=False, indent=4)
print(f"💾 Saved {len(samples)} samples to Drive")



💾 Saved 210 samples to Drive


In [12]:

import glob, json

all_files = glob.glob('/content/drive/MyDrive/NCKH_dataset/*_raw.json')
total = 0
for f in all_files:
    with open(f) as fh:
        data = json.load(fh)
        print(f"  {os.path.basename(f)}: {len(data)} samples")
        total += len(data)

  curl_raw.json: 210 samples


In [14]:
import json

# Load file dataset lên từ Drive
with open('/content/drive/MyDrive/NCKH_dataset/curl_raw.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

# Định nghĩa hàm tìm kiếm theo mã id
def xem_mau_theo_id(ma_id):
    found = [item for item in dataset if item.get('id') == ma_id]

    if not found:
        print(f"❌ Không tìm thấy mẫu có id là: {ma_id}")
        return

    item = found[0]
    print(f"================ MÃ MẪU: #{item['id']} ================")
    print(f"📁 Dự án: {item['project']} | File: {item['file']}")
    print(f"🔗 Commit: {item['commit']}")
    print("\n---------------- [ 🔴 CODE LÚC CÒN LỖI (BUGGY) ] ----------------")
    print(item['buggy'])
    print("\n---------------- [ 🟢 CODE SAU KHI VÁ (FIXED) ] ----------------")
    print(item['fixed'])
    print("================================================================\n")

print("✅ Đã tải xong dataset và sẵn sàng tra cứu!")

✅ Đã tải xong dataset và sẵn sàng tra cứu!


In [16]:
# Ví dụ muốn xem mã "001" hay "005" thì bạn cứ gõ vào đây rồi bấm chạy:
xem_mau_theo_id("010")

================ MÃ MẪU: #010 ================
📁 Dự án: curl | File: lib/url.c
🔗 Commit: 1a17959fc7a80864c9c9b37097856bdb4a465cf8

---------------- [ 🔴 CODE LÚC CÒN LỖI (BUGGY) ] ----------------
/***************************************************************************
 *                                  _   _ ____  _
 *  Project                     ___| | | |  _ \| |
 *                             / __| | | | |_) | |
 *                            | (__| |_| |  _ <| |___
 *                             \___|\___/|_| \_\_____|
 *
 * Copyright (C) Daniel Stenberg, <daniel@haxx.se>, et al.
 *
 * This software is licensed as described in the file COPYING, which
 * you should have received as part of this distribution. The terms
 * are also available at https://curl.se/docs/copyright.html.
 *
 * You may opt to use, copy, modify, merge, publish, distribute and/or sell
 * copies of the Software, and permit persons to whom the Software is
 * furnished to do so, under the terms of the COPYING